# Production Research Pipeline

This is the canonical notebook front door. Training, inference, losses, metrics, and data loading live in `src/` and `scripts/`; this notebook should stay thin.

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from research_config import ALL_SEGMENTS, TrainConfig
from ink_models import build_model

print('Segments:', len(ALL_SEGMENTS))
print('Default val segment:', TrainConfig().val_segment)

## Smoke Check

This checks that both standardized model definitions produce dense ink logits.

In [ ]:
import torch

for name in ['baseline', 'unet_v2']:
    model = build_model(name).eval()
    x = torch.randn(1, 33, 32, 32)
    with torch.no_grad():
        y = model(x)
    print(name, tuple(y.shape), f'{sum(p.numel() for p in model.parameters())/1e6:.2f}M params')

## Recommended Runs

Run these from the repository root in PowerShell. Keep the validation segment fixed when comparing label sources.

In [ ]:
print(r'.\venv\Scripts\python.exe scripts\train_segment_model.py --model unet_v2')
print(r'.\venv\Scripts\python.exe scripts\train_segment_model.py --model unet_v2 --label-root predictions\improved_labels_visual')
print(r'.\venv\Scripts\python.exe scripts\summarize_runs.py')

## Notebook Policy

- Use this notebook for orchestration and sanity checks.
- Use visualization notebooks for inspection only.
- Use old model notebooks as ablation references, not as the source of truth.
- Any model comparison should come from `models/runs/*/history.json`.